In [1]:
!pip install kaggle


In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"kagglecomabubakar","key":"1ec7345524fb7ca73567796c2be50455"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!pip install opendatasets

In [6]:
from opendatasets import download
download("https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification")

Skipping, found downloaded files in "./fake-news-classification" (use force=True to force download)


In [6]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [4]:
import os
data_path = os.listdir('/content/fake-news-classification')
data_path

['WELFake_Dataset.csv']

In [7]:
df = pd.read_csv("/content/fake-news-classification/WELFake_Dataset.csv")
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [8]:
df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)

In [17]:
df_small = df.sample(n=30000, random_state=42)



In [18]:
df = df_small

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30000 entries, 2308 to 23654
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   29770 non-null  object
 1   text    29982 non-null  object
 2   label   30000 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 937.5+ KB


In [20]:
df.isnull().sum()

,0
title,230
text,18
label,0


In [21]:
df['label'].value_counts()

,count
label,
1,15291
0,14709


In [22]:
df.shape

(30000, 3)

In [23]:
df = df.dropna() #Handled Missing values by droping those rows

In [24]:
df.isna().sum()

,0
title,0
text,0
label,0


In [25]:
df.shape

(29752, 3)

In [26]:
df.reset_index(inplace=True)
df.head()

,index,title,text,label
0,2308,Trump says fresh North Korea sanctions 'nothin...,WASHINGTON (Reuters) - U.S. President Donald T...,0
1,22404,CAN IT GET MORE CORRUPT? Bill That Bans Naming...,Editor’s Note : Disgusting. This country just ...,1
2,23397,"U.S. pressure or not, U.N. nuclear watchdog se...",VIENNA (Reuters) - The United States is pushin...,0
3,25058,SERIOUSLY INJURED Cop Sues Black Lives Matter…...,This group of Black Lives Matter supporters ma...,1
4,2664,WHY MUSLIM IMMIGRANT Welfare Fraud Has Explode...,"The cases of fraud, money laundering and theft...",1


In [27]:
# df.drop('Unnamed: 0', axis=1, inplace=True)
# df.head()

In [28]:
df['title'][0]

"Trump says fresh North Korea sanctions 'nothing' compared to what needs to happen"

In [29]:
df = df.drop(['id','text'],axis = 1)
df.head()

KeyError: "['id'] not found in axis"

## **Data Preprocessing**

# 1.Tokenization


In [30]:
sample_data = 'The quick brown fox jumps over the lazy dog'
sample_data = sample_data.split()
sample_data

['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

# 2. Make Lowercase

In [31]:
sample_data = [data.lower() for data in sample_data]
sample_data

['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

# 3. Remove Stopwords

In [32]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_r

True

In [34]:
import nltk
from nltk.corpus import stopwords

# make sure stopwords corpus is downloaded
nltk.download('stopwords')

# get the list of English stopwords
english_stopwords = stopwords.words('english')

# print them all
print(english_stopwords)

# if you want to see how many there are
print("Total:", len(english_stopwords))



['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [35]:
sample_data = [data for data in sample_data if data not in english_stopwords]
print(sample_data)
len(sample_data)

['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']


6

#4. Stemming

In [36]:
ps = PorterStemmer()
sample_data_stemming = [ps.stem(data) for data in sample_data]
print(sample_data_stemming)

['quick', 'brown', 'fox', 'jump', 'lazi', 'dog']


#5. Lemmatization

In [37]:
lm = WordNetLemmatizer()
sample_data_lemma = [lm.lemmatize(data) for data in sample_data]
print(sample_data_lemma)

['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']


In [38]:
lm = WordNetLemmatizer()
corpus = []
for i in range (len(df)):
    review = re.sub('^a-zA-Z0-9',' ', df['title'][i])
    review = review.lower()
    review = review.split()
    review = [lm.lemmatize(x) for x in review if x not in english_stopwords]
    review = " ".join(review)
    corpus.append(review)

In [39]:
len(corpus)

29752

In [40]:
corpus[1]


'get corrupt? bill ban naming officer involved shooting go pennsylvania governor'

In [41]:
df['title'][1]

'CAN IT GET MORE CORRUPT? Bill That Bans Naming Officers Involved in Shootings Goes to Pennsylvania Governor'

## 4. Vectorization (Convert Text data into the Vector)

In [42]:
tf = TfidfVectorizer()
x = tf.fit_transform(corpus).toarray()
x

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [43]:
y = df['label']
y.head()

,label
0,0
1,1
2,0
3,1
4,1


# **Data splitting into the train and test**

In [44]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.3, random_state = 10, stratify = y )

In [45]:
len(x_train),len(y_train)

(20826, 20826)

In [46]:
len(x_test), len(y_test)

(8926, 8926)

## **5. Model Building**

In [47]:
rf = RandomForestClassifier()
rf.fit(x_train, y_train)

RandomForestClassifier()

# **6. Model Evaluation**

In [48]:
y_pred = rf.predict(x_test)
accuracy_score_ = accuracy_score(y_test,y_pred)
accuracy_score_

0.8849428635447009

In [52]:
class Evaluation:

    def __init__(self,model,x_train,x_test,y_train,y_test):
        self.model = model
        self.x_train = x_train
        self.x_test = x_test
        self.y_train = y_train
        self.y_test = y_test

    def train_evaluation(self):
        y_pred_train = self.model.predict(self.x_train)

        acc_scr_train = accuracy_score(self.y_train,y_pred_train)
        print("Accuracy Score On Training Data Set :",acc_scr_train)
        print()

        con_mat_train = confusion_matrix(self.y_train,y_pred_train)
        print("Confusion Matrix On Training Data Set :\n",con_mat_train)
        print()

        class_rep_train = classification_report(self.y_train,y_pred_train)
        print("Classification Report On Training Data Set :\n",class_rep_train)


    def test_evaluation(self):
        y_pred_test = self.model.predict(self.x_test)

        acc_scr_test = accuracy_score(self.y_test,y_pred_test)
        print("Accuracy Score On Testing Data Set :",acc_scr_test)
        print()

        con_mat_test = confusion_matrix(self.y_test,y_pred_test)
        print("Confusion Matrix On Testing Data Set :\n",con_mat_test)
        print()

        class_rep_test = classification_report(self.y_test,y_pred_test)
        print("Classification Report On Testing Data Set :\n",class_rep_test)

In [50]:
#Checking the accuracy on training dataset

Evaluation(rf,x_train, x_test, y_train, y_test).train_evaluation()

Accuracy Score On Training Data Set : 0.9999519830980506

Confusion Matrix On Training Data Set :
 [[10295     1]
 [    0 10530]]

Classification Report On Training Data Set :
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     10296
           1       1.00      1.00      1.00     10530

    accuracy                           1.00     20826
   macro avg       1.00      1.00      1.00     20826
weighted avg       1.00      1.00      1.00     20826



In [53]:
#Checking the accuracy on testing dataset
Evaluation(rf,x_train, x_test, y_train, y_test).test_evaluation()

Accuracy Score On Testing Data Set : 0.8849428635447009

Confusion Matrix On Testing Data Set :
 [[3824  589]
 [ 438 4075]]

Classification Report On Testing Data Set :
               precision    recall  f1-score   support

           0       0.90      0.87      0.88      4413
           1       0.87      0.90      0.89      4513

    accuracy                           0.88      8926
   macro avg       0.89      0.88      0.88      8926
weighted avg       0.89      0.88      0.88      8926



## **Prediction Pipeline**

In [57]:
class Preprocessing:

    def __init__(self,data):
        self.data = data

    def text_preprocessing_user(self):
        lm = WordNetLemmatizer()
        pred_data = [self.data]
        preprocess_data = []
        for data in pred_data:
            review = re.sub('^a-zA-Z0-9',' ', data)
            review = review.lower()
            review = review.split()
            review = [lm.lemmatize(x) for x in review if x not in english_stopwords]
            review = " ".join(review)
            preprocess_data.append(review)
        return preprocess_data

In [58]:
df['title'][1]

'CAN IT GET MORE CORRUPT? Bill That Bans Naming Officers Involved in Shootings Goes to Pennsylvania Governor'

In [64]:
data = 'CAN IT GET MORE CORRUPT? Bill That Bans Naming Officers Involved in Shootings Goes to Pennsylvania Governor'
Preprocessing(data).text_preprocessing_user()

['get corrupt? bill ban naming officer involved shooting go pennsylvania governor']

In [60]:
class Prediction:

    def __init__(self,pred_data, model):
        self.pred_data = pred_data
        self.model = model

    def prediction_model(self):
        preprocess_data = Preprocessing(self.pred_data).text_preprocessing_user()
        data = tf.transform(preprocess_data)
        prediction = self.model.predict(data)

        if prediction [0] == 0 :
            return "The News Is Fake"

        else:
            return "The News Is Real"


In [61]:
data = 'FLYNN: Hillary Clinton, Big Woman on Campus - Breitbart'
Prediction(data,rf).prediction_model()

'The News Is Fake'

In [62]:
df['title'][3]

'SERIOUSLY INJURED Cop Sues Black Lives Matter…Does He Have A Case?'

In [63]:
user_data = '15 Civilians Killed In Single US Airstrike Have Been Identified'
Prediction(user_data,rf).prediction_model()

'The News Is Real'